In [1]:
import pandas as pd
from unidecode import unidecode
import re
from collections import Counter

In [2]:
users_df = pd.read_csv("data/fusers.csv")
users_df = users_df[["id", "name", "screen_name"]].copy()

wine_df = pd.read_csv("data/full_wine_dataset.csv")

In [3]:
users_df.head()

,id,name,screen_name
0,370098498,pirfectmoses,pirfectmoses
1,37384589,SAK Nair,bsknair1967
2,72110028,Deepak,dedjven
3,82885728,Marcos Vinicius,BrowAlves
4,110120789,Shri Kant Kanaujia,kanaujiask


## Inspect wine_df

In [4]:
wine_df.head()

,href,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
0,https://www.danmurphys.com.au/product/DM_17275...,Bin 128 Shiraz 2019,Enjoy layers of dark fruit and savoury spice b...,Red,Shiraz,2019,South Australia,Coonawarra,Peter Gago,750mL,50.20,NaN,Full Bodied,Dry
1,https://www.danmurphys.com.au/product/DM_90496...,Penfolds Koonunga Hill Shiraz 2021 750mL,A multi-regional blend of Shiraz from across S...,Red,Shiraz,NaN,South Australia,Multi-Regional Blend,NaN,750mL,12.95,11.65,Full Bodied,Dry
2,https://www.danmurphys.com.au/product/DM_22865...,Estate Shiraz,"At release, this Shiraz is a dark red colour w...",Red,Shiraz,2019,South Australia,Multi-Regional Blend,NaN,750mL,15.95,NaN,Medium-bodied,Dry
3,https://www.danmurphys.com.au/product/DM_54131...,Shiraz,19 Crimes turned criminals into colonists. Upo...,Red,Shiraz,NaN,South Australia,Multi-Regional Blend,NaN,750mL,10.20,NaN,Full Bodied,Light Bodied
4,https://www.danmurphys.com.au/product/DM_72304...,Barossa Cabernet Sauvignon,Cat Amongst The Pigeons has been ruffling a fe...,Red,Cabernet Sauvignon,2022*,South Australia,Barossa Valley,NaN,750mL,14.95,NaN,Full Bodied,Dry


In [5]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1310 entries, 0 to 1309
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   href                 1310 non-null   object 
 1   product_name         1309 non-null   object 
 2   product_description  1298 non-null   object 
 3   colour               1310 non-null   object 
 4   type                 1256 non-null   object 
 5   vintage              1135 non-null   object 
 6   state                830 non-null    object 
 7   region               1076 non-null   object 
 8   winery               478 non-null    object 
 9   size                 1287 non-null   object 
 10  non_member_price     1309 non-null   float64
 11  member_price         213 non-null    float64
 12  body                 953 non-null    object 
 13  sweetness            973 non-null    object 
dtypes: float64(2), object(12)
memory usage: 143.4+ KB


In [6]:
wine_df.type.value_counts()

type
Shiraz                                         216
Chardonnay                                     121
Cabernet Sauvignon                             104
Pinot Noir                                      77
Sauvignon Blanc                                 64
                                              ... 
Marzemino                                        1
Pinot Nero                                       1
Nero DÀvola                                      1
Lambrusco                                        1
Pinot Noir & Meunier (75%) Chardonnay (25%)      1
Name: count, Length: 269, dtype: int64

## Clean type column

In [7]:
# drop null values
wine_df = wine_df.dropna(subset=["type"]).reset_index(drop=True)

In [8]:
# lowercase and remove accents
def clean_characters(type_str):
    if isinstance(type_str, str):  # Check if the value is a string
        return unidecode(type_str).lower()
    else:
        return type_str

# Apply the function to the 'type' column
wine_df['type'] = wine_df['type'].apply(clean_characters)

In [9]:
wine_df.type.value_counts()

type
shiraz                                         216
chardonnay                                     122
cabernet sauvignon                             105
pinot noir                                      79
sauvignon blanc                                 64
                                              ... 
marzemino                                        1
pinot nero                                       1
nero davola                                      1
lambrusco                                        1
pinot noir & meunier (75%) chardonnay (25%)      1
Name: count, Length: 258, dtype: int64

In [10]:
# remove non-words
def filter_words(type_str):
    if isinstance(type_str, str):
        # Remove any characters that are not letters or spaces
        cleaned_text = re.sub(r'[^a-zA-Z\s]', '', type_str)
        # Remove extra spaces and convert
        return ' '.join(cleaned_text.split())
    else:
        return type_str

wine_df['type'] = wine_df['type'].apply(filter_words)

In [11]:
# inspect the remaining type values
pd.set_option('display.max_rows', None)  # Set to None to display all rows
type_counts = wine_df['type'].value_counts()
type_counts_with_index = pd.DataFrame({'Wine Type': type_counts.index, 'Count': type_counts.values})
print(type_counts_with_index)
pd.reset_option('display.max_rows')

                                             Wine Type  Count
0                                               shiraz    217
1                                           chardonnay    123
2                                   cabernet sauvignon    105
3                                           pinot noir     79
4                                      sauvignon blanc     64
5                                             riesling     41
6                                                 rose     41
7                                               merlot     26
8                                pinot noir chardonnay     25
9                                             grenache     22
10                                         tempranillo     21
11                                        pinot grigio     20
12                                     cabernet merlot     17
13                                          pinot gris     17
14                                            prosecco     14
15      

### rebuild type column

I will retrieve idividual type names from the data and use them to filter out ambiguous or sparse types. Finally I will rebuild the types to be more consistent in how they represent blends.

In [12]:
# manually retrieve individual wine types to create a type list
indices_to_retrieve = list(range(0, 8))  + list(range(9, 12)) + list(range(13, 16)) + [17] + list(range(22, 24)) \
    + list(range(26, 28)) + [29] + [31] + list(range(33, 37)) + list(range(39, 44)) + [49]
types_list = list(wine_df.type.value_counts().index)
filtered_types_list = [types_list[i] for i in indices_to_retrieve]
filtered_types_list.append("garnacha")
filtered_types_list.sort()
filtered_types_list = filtered_types_list + ["white", "red", "blend"] # add generic terms
filtered_types_list = ["sparkling"] + filtered_types_list # add generic terms
print(filtered_types_list)

['sparkling', 'cabernet sauvignon', 'chardonnay', 'chenin blanc', 'fiano', 'gamay', 'garnacha', 'glera', 'grenache', 'grillo', 'gruner veltliner', 'malbec', 'merlot', 'moscato', 'nebbiolo', 'nero davola', 'pet nat', 'pinot grigio', 'pinot gris', 'pinot noir', 'prosecco', 'rice wine', 'riesling', 'rose', 'sangiovese', 'sauvignon blanc', 'semillon', 'shiraz', 'syrah', 'tempranillo', 'verdelho', 'vermentino', 'viognier', 'white', 'red', 'blend']


In [13]:
# save filtered wine types
#pd.DataFrame({"type":filtered_types_list}).to_csv('filtered_wine_types.csv', index=False)

#### convert pinot grisgrigio

In [14]:
# convert pinot grigiogris to pinot gris or pinot grigio
wine_df[wine_df["type"] == "pinot grigiogris"]

,href,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
913,https://www.danmurphys.com.au/product/DM_22943...,White Mist Pinot Gris,The South Island of New Zealand is home to Mar...,White,pinot grigiogris,2022*,NaN,Marlborough,NaN,750mL,17.29,NaN,Light Bodied,Dry
917,https://www.danmurphys.com.au/product/DM_21457...,Pinot Gris 750mL,"Like the Paradox of life, an infinate mystery,...",White,pinot grigiogris,2022,NaN,South Eastern Australia,NaN,750mL,13.29,11.0,Light Bodied,Dry
1131,https://www.danmurphys.com.au/product/DM_21480...,Portia's 'Big Dog' Pinot Grigio,"Proudly Australian, Sit Stay Society celebrate...",White,pinot grigiogris,2022*,NaN,FLEURIEU Peninsula,NaN,750mL,16.29,13.0,Light Bodied,Dry


In [15]:
# replace pinot grisgrigio
wine_df.loc[wine_df["product_name"] == "White Mist Pinot Gris", "type"] = "pinot gris"
wine_df.loc[wine_df["product_name"] == "Pinot Gris 750mL", "type"] = "pinot gris"
wine_df.loc[wine_df["product_name"] == "Portia's 'Big Dog' Pinot Grigio", "type"] = "pinot grigio"

In [16]:
# convert pinot grigiogris to pinot gris or pinot grigio
wine_df[wine_df["type"] == "pinot grigiogris"]

,href,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness


#### Clean cabernet type values

In [17]:
len(wine_df[wine_df["type"].str.contains("cabernet sauvignon")])

133

In [18]:
wine_df['type'] = wine_df['type'].str.replace("cabernet sauvignon", 'cabernet', regex=True)
wine_df['type'] = wine_df['type'].str.replace("cabernet", 'cabernet sauvignon', regex=True)

In [19]:
len(wine_df[wine_df["type"].str.contains("cabernet sauvignon")])

186

#### start matching types

In [20]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1256 entries, 0 to 1255
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   href                 1256 non-null   object 
 1   product_name         1256 non-null   object 
 2   product_description  1245 non-null   object 
 3   colour               1256 non-null   object 
 4   type                 1256 non-null   object 
 5   vintage              1091 non-null   object 
 6   state                811 non-null    object 
 7   region               1052 non-null   object 
 8   winery               470 non-null    object 
 9   size                 1244 non-null   object 
 10  non_member_price     1256 non-null   float64
 11  member_price         207 non-null    float64
 12  body                 943 non-null    object 
 13  sweetness            961 non-null    object 
dtypes: float64(2), object(12)
memory usage: 137.5+ KB


In [21]:
wine_df.iloc[1227]

href                   https://www.danmurphys.com.au/product/DM_ER_20...
product_name                                        Sparkling Brut Cuvee
product_description    Selected fruit from premium South Australian v...
colour                                               Champagne-Sparkling
type                                                          brut cuvee
vintage                                                      Non Vintage
state                                                    South Australia
region                                                   Langhorne Creek
winery                                                     Peter Pollard
size                                                               750ML
non_member_price                                                    20.0
member_price                                                         NaN
body                                                        Light Bodied
sweetness                                          

In [22]:
"""all_matching_words = []
leftover_words = []
for i, row in wine_df.iterrows():
    matched = False
    # build matched types list
    row_matching_words = []
    for word in filtered_types_list:
        if word in row['type']:
            row_matching_words.append(word)
            matched = True

    # check for extra words
    if matched:
        row_type = row.copy()['type']
        for matched_word in row_matching_words:
            row_type = row_type.replace(matched_word, "_")
        remaining_words = re.findall(r'\b[a-zA-Z]+\b', row_type)
        if (len(remaining_words) > 0 or len(row_matching_words) > 1) and "blend" not in row_matching_words:
            row_matching_words.append("blend")
    else:
        row_type = row.copy()['type']
        remaining_words = []

    leftover_words.append(remaining_words)
    print(i, row_matching_words, row_type, remaining_words)
    print(i, row['type'])
    all_matching_words.append(row_matching_words)"""

'all_matching_words = []\nleftover_words = []\nfor i, row in wine_df.iterrows():\n    matched = False\n    # build matched types list\n    row_matching_words = []\n    for word in filtered_types_list:\n        if word in row[\'type\']:\n            row_matching_words.append(word)\n            matched = True\n\n    # check for extra words\n    if matched:\n        row_type = row.copy()[\'type\']\n        for matched_word in row_matching_words:\n            row_type = row_type.replace(matched_word, "_")\n        remaining_words = re.findall(r\'\x08[a-zA-Z]+\x08\', row_type)\n        if (len(remaining_words) > 0 or len(row_matching_words) > 1) and "blend" not in row_matching_words:\n            row_matching_words.append("blend")\n    else:\n        row_type = row.copy()[\'type\']\n        remaining_words = []\n\n    leftover_words.append(remaining_words)\n    print(i, row_matching_words, row_type, remaining_words)\n    print(i, row[\'type\'])\n    all_matching_words.append(row_matching_wo

In [23]:
all_matching_words = []
leftover_words = []
for i, row in wine_df.iterrows():
    matched = False
    # build matched types list
    row_matching_words = []
    for word in filtered_types_list:
        if word in row['type']:
            row_matching_words.append(word)
            matched = True

    # check if blend
    row_type = row.copy()['type']
    for matched_word in row_matching_words:
        row_type = row_type.replace(matched_word, "_")
    remaining_words = re.findall(r'\b[a-zA-Z]+\b', row_type)
    if (len(remaining_words) > 0 or len(row_matching_words) > 1) and "blend" not in row_matching_words and matched:
        row_matching_words.append("blend")

    leftover_words.append(remaining_words)
    print(i, row_matching_words, row_type, remaining_words)
    print(i, row['type'])
    all_matching_words.append(row_matching_words)

0 ['shiraz'] _ []
0 shiraz
1 ['shiraz'] _ []
1 shiraz
2 ['shiraz'] _ []
2 shiraz
3 ['shiraz'] _ []
3 shiraz
4 ['cabernet sauvignon'] _ []
4 cabernet sauvignon
5 ['rose'] _ []
5 rose
6 ['cabernet sauvignon'] _ []
6 cabernet sauvignon
7 ['cabernet sauvignon'] _ []
7 cabernet sauvignon
8 ['cabernet sauvignon', 'shiraz', 'blend'] _ _ _ _ []
8 shiraz cabernet sauvignon cabernet sauvignon shiraz
9 ['shiraz'] _ []
9 shiraz
10 ['cabernet sauvignon'] _ []
10 cabernet sauvignon
11 ['merlot'] _ []
11 merlot
12 ['merlot'] _ []
12 merlot
13 ['cabernet sauvignon', 'merlot', 'blend'] _ _ []
13 cabernet sauvignon merlot
14 ['shiraz'] _ []
14 shiraz
15 ['pinot noir'] _ []
15 pinot noir
16 ['merlot'] _ []
16 merlot
17 ['pinot noir'] _ []
17 pinot noir
18 ['pinot noir'] _ []
18 pinot noir
19 ['cabernet sauvignon'] _ []
19 cabernet sauvignon
20 ['shiraz'] _ []
20 shiraz
21 ['shiraz'] _ []
21 shiraz
22 ['cabernet sauvignon'] _ []
22 cabernet sauvignon
23 [] dolcetto ['dolcetto']
23 dolcetto
24 ['shiraz'] _

#### investigate leftover type words

In [24]:
# investigate leftover words

# Flatten the list of lists
flat_list = [item for sublist in leftover_words for item in sublist]

# Count the occurrences of each string
word_counts = Counter(flat_list)

# Display the word counts
print(word_counts)

Counter({'other': 22, 'pinot': 19, 'meunier': 15, 'and': 11, 'wine': 9, 'sbs': 9, 'franc': 8, 'ssb': 8, 'mourvedre': 5, 'petit': 4, 'verdot': 4, 'muscat': 4, 'blanc': 4, 'brut': 4, 'mataro': 3, 'gsm': 3, 'noir': 3, 'vranec': 3, 'gewurztraminer': 3, 'msg': 2, 'sgm': 2, 'agiorgitiko': 2, 'carignan': 2, 'burgundy': 2, 'zinfandel': 2, 'touriga': 2, 'field': 2, 'cinsault': 2, 'varieties': 2, 'varietal': 2, 'sweet': 2, 'durif': 2, 'graciano': 2, 'monastrell': 2, 'saperavi': 2, 'fino': 2, 'roussanne': 2, 'pecorino': 2, 'picpoul': 2, 'de': 2, 'cuvee': 2, 'dolcetto': 1, 'montepulciano': 1, 'primitivo': 1, 'sauvingon': 1, 'pinotage': 1, 'grape': 1, 'of': 1, 'veneto': 1, 'mouuvedre': 1, 'chianti': 1, 'sangiovesse': 1, 'mixed': 1, 'brachetto': 1, 'merot': 1, 'bunny': 1, 'refosco': 1, 'sherry': 1, 'mondeuse': 1, 'roussane': 1, 'piont': 1, 'assyrtiko': 1, 'rkatsiteli': 1, 'cannonau': 1, 'raboso': 1, 'marzemino': 1, 'nero': 1, 'lambrusco': 1, 'limniona': 1, 'semidry': 1, 'gris': 1, 'maturana': 1, 'bl

In [25]:
leftover_words

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['dolcetto'],
 [],
 ['mataro'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['gsm', 'msg', 'sgm'],
 ['other'],
 ['agiorgitiko'],
 [],
 [],
 [],
 [],
 ['montepulciano'],
 [],
 [],
 [],
 [],
 [],
 [],
 ['other'],
 [],
 [],
 [],
 [],
 ['gsm', 'msg', 'sgm'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['other'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['pinot'],
 [],
 [],
 [],
 [],
 [],
 ['franc'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['primitivo'],
 [],
 [],
 [],
 [],
 [],
 ['noir', 'carignan'],
 [],
 [],
 [],
 [],
 [],
 [],
 ['other'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['agiorgitiko'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['sauvingon'],
 [],
 [],
 [],
 ['burgundy'],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 

In [26]:
wine_df["leftover_words"] = leftover_words

In [27]:
wine_df[wine_df["leftover_words"].apply(lambda x: len(x) > 0)]

,href,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words
23,https://www.danmurphys.com.au/product/DM_55398...,Multi-Regional Blend Dolcetto & Syrah,Brown Brothers Dolcetto & Syrah is a firm favo...,Red,dolcetto,NaN,Victoria,Multi-Regional Blend,NaN,750mL,12.90,NaN,Medium Bodied,Sweet,[dolcetto]
25,https://www.danmurphys.com.au/product/DM_67338...,Barossa Grenache Shiraz Mataró,"Seek out these wines for simple, everyday drin...",Red,grenache shiraz mataro,2022*,South Australia,Barossa Valley,NaN,750ML,14.95,NaN,Full Bodied,Dry,[mataro]
35,https://www.danmurphys.com.au/product/DM_90136...,The Stump Jump Grenache Mourvedre Shiraz,d'Arenberg are quintessential McLaren Vale and...,Red,gsm msg sgm,NaN,South Australia,McLaren Vale,Chester Osborn,750mL,13.29,NaN,Medium Bodied,Dry,"[gsm, msg, sgm]"
36,https://www.danmurphys.com.au/product/DM_99940...,Red et Al Grenache Shiraz Nero,Bush vine Grenache stars in this original blen...,Red,other red blend,2020*,South Australia,McLaren Vale,Alex Sherrah,750mL,30.29,NaN,Medium Bodied,Dry,[other]
37,https://www.danmurphys.com.au/product/DM_98984...,Nemea Agiorgitiko 2017,"Rich and structured, it is balanced between fr...",Red,agiorgitiko,2019,NaN,Peloponnese,Yiannis Paraskevopoulous,750mL,38.99,NaN,Medium Bodied,NaN,[agiorgitiko]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237,https://www.danmurphys.com.au/product/DM_ER_10...,Pinot Chardonnay Cuvee NV,Made in the traditional Methode Champenoise st...,Champagne-Sparkling,sparkling wine,Non Vintage,Victoria,Alpine Valleys,Matt Kilby,750mL,150.00,NaN,NaN,Dry,[wine]
1240,https://www.danmurphys.com.au/product/DM_ER_10...,"Water Fairies, Tasmanian Vintage Brut 2010","With Jubilation, the encased fairies dance acr...",Champagne-Sparkling,brut,2010,Tasmania,NaN,Brad Rey,750mL,510.00,NaN,Light Bodied,Dry,[brut]
1242,https://www.danmurphys.com.au/product/DM_ER_10...,Docg,"Asti DOCG Five Stars\n\nSweet, 100% from white...",Champagne-Sparkling,muscate,Non Vintage,NaN,Piemonte,Bosca SPA,750mL,108.00,NaN,Light Bodied,Sweet,[muscate]
1244,https://www.danmurphys.com.au/product/DM_ER_10...,Do Cava Gran Reserva De La Finca 2019,Toasty and generous with minerals and dried fl...,Champagne-Sparkling,parellada macabeo xa,2019,WNE,DO Cava,NaN,750mL,544.60,NaN,NaN,Dry,"[parellada, macabeo, xa]"


In [28]:
wine_df.iloc[1227]

href                   https://www.danmurphys.com.au/product/DM_ER_20...
product_name                                        Sparkling Brut Cuvee
product_description    Selected fruit from premium South Australian v...
colour                                               Champagne-Sparkling
type                                                          brut cuvee
vintage                                                      Non Vintage
state                                                    South Australia
region                                                   Langhorne Creek
winery                                                     Peter Pollard
size                                                               750ML
non_member_price                                                    20.0
member_price                                                         NaN
body                                                        Light Bodied
sweetness                                          

#### add matching type words

In [29]:
# add type list as column
wine_df["type_as_list"] = all_matching_words

# re-write type column
wine_df['type'] = wine_df['type_as_list'].apply(lambda x: " ".join(x))

#remove rows with empty type value lists
wine_df = wine_df[wine_df["type_as_list"].apply(lambda x: len(x) > 0)].reset_index(drop=True)

In [30]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1173 entries, 0 to 1172
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   href                 1173 non-null   object 
 1   product_name         1173 non-null   object 
 2   product_description  1163 non-null   object 
 3   colour               1173 non-null   object 
 4   type                 1173 non-null   object 
 5   vintage              1017 non-null   object 
 6   state                785 non-null    object 
 7   region               989 non-null    object 
 8   winery               444 non-null    object 
 9   size                 1162 non-null   object 
 10  non_member_price     1173 non-null   float64
 11  member_price         197 non-null    float64
 12  body                 891 non-null    object 
 13  sweetness            902 non-null    object 
 14  leftover_words       1173 non-null   object 
 15  type_as_list         1173 non-null   o

In [31]:
wine_df.head()

,href,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness,leftover_words,type_as_list
0,https://www.danmurphys.com.au/product/DM_17275...,Bin 128 Shiraz 2019,Enjoy layers of dark fruit and savoury spice b...,Red,shiraz,2019,South Australia,Coonawarra,Peter Gago,750mL,50.20,NaN,Full Bodied,Dry,[],[shiraz]
1,https://www.danmurphys.com.au/product/DM_90496...,Penfolds Koonunga Hill Shiraz 2021 750mL,A multi-regional blend of Shiraz from across S...,Red,shiraz,NaN,South Australia,Multi-Regional Blend,NaN,750mL,12.95,11.65,Full Bodied,Dry,[],[shiraz]
2,https://www.danmurphys.com.au/product/DM_22865...,Estate Shiraz,"At release, this Shiraz is a dark red colour w...",Red,shiraz,2019,South Australia,Multi-Regional Blend,NaN,750mL,15.95,NaN,Medium-bodied,Dry,[],[shiraz]
3,https://www.danmurphys.com.au/product/DM_54131...,Shiraz,19 Crimes turned criminals into colonists. Upo...,Red,shiraz,NaN,South Australia,Multi-Regional Blend,NaN,750mL,10.20,NaN,Full Bodied,Light Bodied,[],[shiraz]
4,https://www.danmurphys.com.au/product/DM_72304...,Barossa Cabernet Sauvignon,Cat Amongst The Pigeons has been ruffling a fe...,Red,cabernet sauvignon,2022*,South Australia,Barossa Valley,NaN,750mL,14.95,NaN,Full Bodied,Dry,[],[cabernet sauvignon]


In [32]:
wine_df.type.value_counts()

type
shiraz                            217
chardonnay                        123
cabernet sauvignon                106
pinot noir                         79
sauvignon blanc                    64
                                 ... 
cabernet sauvignon syrah blend      1
garnacha tempranillo blend          1
malbec shiraz blend                 1
nebbiolo blend                      1
chardonnay shiraz blend             1
Name: count, Length: 93, dtype: int64

### Explore value counts of type

In [33]:
type_counts_df = pd.DataFrame(wine_df.type.value_counts())

In [34]:
type_counts_df.to_csv("data/wine_types.csv")

In [35]:
type_counts_df[type_counts_df["count"] > 2]['count'].sum()

1114

In [36]:
type_counts_df[type_counts_df["count"] > 2]['count']

type
shiraz                             217
chardonnay                         123
cabernet sauvignon                 106
pinot noir                          79
sauvignon blanc                     64
chardonnay pinot noir blend         44
riesling                            41
rose                                41
red blend                           35
merlot                              26
cabernet sauvignon merlot blend     24
cabernet sauvignon shiraz blend     24
grenache                            23
pinot grigio                        21
tempranillo                         21
pinot gris                          19
sauvignon blanc semillon blend      19
cabernet sauvignon blend            16
prosecco rose blend                 15
moscato                             14
white blend                         13
grenache shiraz blend               13
sangiovese                          13
nero davola                          8
malbec                               8
nebbiolo            